In [124]:
import numpy as np

In [125]:
#functions of non-linear activations
def f_sigmoid(X, deriv=False):
    if not deriv:
        return 1 / (1 + np.exp(-X))
    else:
        return f_sigmoid(X)*(1 - f_sigmoid(X))


def f_softmax(X):
    Z = np.sum(np.exp(X), axis=1)
    Z = Z.reshape(Z.shape[0], 1)
    return np.exp(X) / Z

def f_relu(X, deriv=False):
    if not deriv:
        # Returns X if X > 0, otherwise returns 0
        return np.maximum(0, X)
    else:
        # Derivative is 1 for positive inputs, 0 for negative/zero inputs
        return (X > 0).astype(float)

In [126]:
import sys
def exit_with_err(err_str):
    print >> sys.stderr, err_str
    sys.exit(1)

In [ ]:
#Functionality of a single hidden layer
class Layer:
    def __init__(self, size, batch_size, is_input=False, is_output=False,
                 activation=f_relu):
        self.is_input = is_input
        self.is_output = is_output

        # Z is the matrix that holds output values
        self.Z = np.zeros((batch_size, size[0]))
        # The activation function is an externally defined function (with a
        # derivative) that is stored here
        self.activation = activation

        # W is the outgoing weight matrix for this layer
        self.W = None
        # S is the matrix that holds the inputs to this layer
        self.S = None
        # D is the matrix that holds the deltas for this layer
        self.D = None
        # Fp is the matrix that holds the derivatives of the activation function
        self.Fp = None

        if not is_input:
            self.S = np.zeros((batch_size, size[0]))
            self.D = np.zeros((batch_size, size[0]))

        if not is_output:
            self.W = np.random.normal(size=size, scale=0.01) # initialize the weights randomly with value from 0 to 0.01

        if not is_input and not is_output:
            self.Fp = np.zeros((size[0], batch_size))

    def forward_propagate(self):
        # Input layer passes data to the first set of weights
        if self.is_input:
            return self.Z.dot(self.W)
        
        # calculate the activation (ReLU or Softmax)
        self.Z = self.activation(self.S)

        # If it is the output layer, return the probabilities directly
        if self.is_output:
            return self.Z
        # else, it is the hidden layer
        else:
            # For hidden layers, we add the bias values here, to avoid the case that sum(w.z) is zero which link the neuron decision back to the original inputs.
            # The bias is added before applying the activation function, this calculation is here, after the activation function above since this is for the next layer, not the current one.
            # column of 1 is added here as the input for the bias weight in the next layer
            self.Z = np.append(self.Z, np.ones((self.Z.shape[0], 1)), axis=1)

            # Calculate and store the derivative Fp at this current layer when we have the inputs self.S already in the memory.
            # this Fp is used in the backward pass, giving the direction to update the weights (increase or decrease) to reduce the lost/error
            self.Fp = self.activation(self.S, deriv=True)
            
            return self.Z.dot(self.W)


In [ ]:
class MultiLayerPerceptron:
    def __init__(self, layer_config, batch_size=100):
        self.layers = []
        self.num_layers = len(layer_config)
        self.minibatch_size = batch_size

        for i in range(self.num_layers-1):
            if i == 0:
                print ("Initializing input layer with size {0}.".format(layer_config[i]))
                # Here, we add an additional unit at the input for the bias
                # weight.
                self.layers.append(Layer([layer_config[i]+1, layer_config[i+1]],
                                         batch_size,
                                         is_input=True))
                print ("No activation for the input layer")
            else:
                print ("Initializing hidden layer with size {0}.".format(layer_config[i]))
                # Here we add an additional unit in the hidden layers for the
                # bias weight.
                self.layers.append(Layer([layer_config[i]+1, layer_config[i+1]],
                                         batch_size,
                                         activation=f_relu))
                print ("ReLU activation for the hinden layer")

        print ("Initializing output layer with size {0}.".format(layer_config[-1]))
        self.layers.append(Layer([layer_config[-1], None],
                                 batch_size,
                                 is_output=True,
                                 activation=f_softmax))
        print ("Softmax activation for the output layer")
        print ("Done!")

    def forward_propagate(self, data):
        # We need to be sure to add bias values to the input
        self.layers[0].Z = np.append(data, np.ones((data.shape[0], 1)), axis=1)

        for i in range(self.num_layers-1):
            self.layers[i+1].S = self.layers[i].forward_propagate()
        return self.layers[-1].forward_propagate()

    def backpropagate(self, yhat, labels):
        
        # exit_with_err("FIND ME IN THE CODE, What is computed in the next line of code?\n")

        self.layers[-1].D = (yhat - labels).T # error in the output layer
        for i in range(self.num_layers-2, 0, -1):
            # We do not calculate deltas for the bias weights
            W_nobias = self.layers[i].W[0:-1, :]
            
            # exit_with_err("FIND ME IN THE CODE, What does this 'for' loop do?\n")
            
            
            self.layers[i].D = W_nobias.dot(self.layers[i+1].D) * self.layers[i].Fp  # error calculated on the remaining layers

    def update_weights(self, eta):
        for i in range(0, self.num_layers-1):
            W_grad = -eta*(self.layers[i+1].D.dot(self.layers[i].Z)).T # the adjustment for the weights is controlled by the learning rate, the error procuded by the layer and the input of it.
            self.layers[i].W += W_grad

    def evaluate(self, train_data, train_labels, test_data, test_labels,
                 num_epochs=70, eta=0.05, eval_train=False, eval_test=True):

        N_train = len(train_labels)*len(train_labels[0])
        N_test = len(test_labels)*len(test_labels[0])

        print ("Training for {0} epochs...".format(num_epochs))
        for t in range(0, num_epochs):
            out_str = "[{0:4d}] ".format(t)

            for b_data, b_labels in zip(train_data, train_labels):
                output = self.forward_propagate(b_data)
                self.backpropagate(output, b_labels)
                
                # exit_with_err("FIND ME IN THE CODE, How does weight update is implemented? What is eta?\n")

                self.update_weights(eta=eta)

            if eval_train:
                errs = 0
                for b_data, b_labels in zip(train_data, train_labels):
                    output = self.forward_propagate(b_data)
                    yhat = np.argmax(output, axis=1)
                    errs += np.sum(1-b_labels[np.arange(len(b_labels)), yhat])

                out_str = ("{0} Training error: {1:.5f}".format(out_str,
                                                           float(errs)/N_train))

            if eval_test:
                errs = 0
                for b_data, b_labels in zip(test_data, test_labels):
                    output = self.forward_propagate(b_data)
                    yhat = np.argmax(output, axis=1)
                    errs += np.sum(1-b_labels[np.arange(len(b_labels)), yhat])

                out_str = ("{0} Test error: {1:.5f}").format(out_str,
                                                       float(errs)/N_test)

            print (out_str)


In [129]:
def label_to_bit_vector(labels, nbits):
    bit_vector = np.zeros((labels.shape[0], nbits))
    for i in range(labels.shape[0]):
        bit_vector[i, labels[i]] = 1.0

    return bit_vector

In [130]:
def create_batches(data, labels, batch_size, create_bit_vector=False):
    N = data.shape[0]
    print ("Batch size {0}, the number of examples {1}.".format(batch_size,N))

    if N % batch_size != 0:
        print ("Warning in create_minibatches(): Batch size {0} does not " \
              "evenly divide the number of examples {1}.".format(batch_size,N))
    chunked_data = []
    chunked_labels = []
    idx = 0
    while idx + batch_size <= N:
        chunked_data.append(data[idx:idx+batch_size, :])
        if not create_bit_vector:
            chunked_labels.append(labels[idx:idx+batch_size])
        else:
            bit_vector = label_to_bit_vector(labels[idx:idx+batch_size], 10)
            chunked_labels.append(bit_vector)

        idx += batch_size

    return chunked_data, chunked_labels


In [131]:
def prepare_for_backprop(batch_size, Train_images, Train_labels, Valid_images, Valid_labels):
    
    print ("Creating data...")
    # split 60000 images in the Train_images into 600 batches, each batches contains 100 images (with the original provided setting)
    batched_train_data, batched_train_labels = create_batches(Train_images, Train_labels,
                                              batch_size,
                                              create_bit_vector=True)
    # similarly, 100 batches of 100 images
    batched_valid_data, batched_valid_labels = create_batches(Valid_images, Valid_labels,
                                              batch_size,
                                              create_bit_vector=True)
    print ("Done!")


    return batched_train_data, batched_train_labels,  batched_valid_data, batched_valid_labels



In [132]:
from keras.datasets import mnist

In [133]:
(Xtr, Ltr), (X_test, L_test)=mnist.load_data() # Load the dataset

Xtr = Xtr.reshape(60000, 784)   #flat 60000 images for trainning
X_test = X_test.reshape(10000, 784) #flat 10000 images for testing
Xtr = Xtr.astype('float32') #change type from uint8 to float32 to avoid overflow and sign errors.
X_test = X_test.astype('float32')
Xtr /= 255  # scale Pixel value [0,255] to [0,1]
X_test /= 255
print(Xtr.shape[0], 'train samples')
print(X_test.shape[0], 'test samples')


60000 train samples
10000 test samples


In [ ]:
batch_size=100 # one batch contain 100 images

train_data, train_labels, valid_data, valid_labels=prepare_for_backprop(batch_size, Xtr, Ltr, X_test, L_test)

# initialize the MLP with 
#   784 inputs representing 784 pixels per image, 
#   100 neuron in the 1st hidden layer
#   100 neron in the 2nd hidden layer
#   10 outputs representing the probability of the image being either 0, 1, ..., 9
mlp = MultiLayerPerceptron(layer_config=[784, 100, 100, 10], batch_size=batch_size)

mlp.evaluate(train_data, train_labels, valid_data, valid_labels,eta=0.000155,
             eval_train=True)

print("Done:)\n")


Creating data...
Batch size 100, the number of examples 60000.
Batch size 100, the number of examples 10000.
Done!
Initializing input layer with size 784.
No activation for the input layer
Initializing hidden layer with size 100.
ReLU activation for the hinden layer
Initializing hidden layer with size 100.
ReLU activation for the hinden layer
Initializing output layer with size 10.
Softmax activation for the output layer
Done!
Training for 70 epochs...
[   0]  Training error: 0.35648 Test error: 0.35890
[   1]  Training error: 0.13077 Test error: 0.13070
[   2]  Training error: 0.10985 Test error: 0.10890
[   3]  Training error: 0.10453 Test error: 0.10400
[   4]  Training error: 0.10968 Test error: 0.10470
[   5]  Training error: 0.10570 Test error: 0.10260
[   6]  Training error: 0.11105 Test error: 0.10580
[   7]  Training error: 0.10048 Test error: 0.09640
[   8]  Training error: 0.09752 Test error: 0.09350
[   9]  Training error: 0.09657 Test error: 0.09600
[  10]  Training error:

/var/folders/_z/tbf6y6ys2d71_q76ldqlnql80000gp/T/ipykernel_29465/1380556644.py:10: RuntimeWarning: overflow encountered in exp
  Z = np.sum(np.exp(X), axis=1)
/var/folders/_z/tbf6y6ys2d71_q76ldqlnql80000gp/T/ipykernel_29465/1380556644.py:12: RuntimeWarning: overflow encountered in exp
  return np.exp(X) / Z
/var/folders/_z/tbf6y6ys2d71_q76ldqlnql80000gp/T/ipykernel_29465/1380556644.py:12: RuntimeWarning: invalid value encountered in divide
  return np.exp(X) / Z


[  24]  Training error: 0.90128 Test error: 0.90200
[  25]  Training error: 0.90128 Test error: 0.90200
[  26]  Training error: 0.90128 Test error: 0.90200
[  27]  Training error: 0.90128 Test error: 0.90200
[  28]  Training error: 0.90128 Test error: 0.90200
[  29]  Training error: 0.90128 Test error: 0.90200
[  30]  Training error: 0.90128 Test error: 0.90200
[  31]  Training error: 0.90128 Test error: 0.90200
[  32]  Training error: 0.90128 Test error: 0.90200
[  33]  Training error: 0.90128 Test error: 0.90200
[  34]  Training error: 0.90128 Test error: 0.90200
[  35]  Training error: 0.90128 Test error: 0.90200
[  36]  Training error: 0.90128 Test error: 0.90200
[  37]  Training error: 0.90128 Test error: 0.90200
[  38]  Training error: 0.90128 Test error: 0.90200
[  39]  Training error: 0.90128 Test error: 0.90200
[  40]  Training error: 0.90128 Test error: 0.90200
[  41]  Training error: 0.90128 Test error: 0.90200
[  42]  Training error: 0.90128 Test error: 0.90200
[  43]  Trai